In [ ]:
!pip install "langchain>=0.2" langchain-community langchain-text-splitters \
            chromadb "sentence-transformers>=2.2" \
            langchain-openai  # only if you’ll use OpenAI
# For Ollama (local LLM)
# brew install ollama  # macOS
# winget install Ollama.Ollama  # Windows
# Then: ollama pull llama3.1:8b


In [ ]:
!pip -q install -U langchain langchain-community langchain-text-splitters chromadb sentence-transformers mistralai


In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentence-transformers \
                  langchain langchain-community langchain-text-splitters chromadb


In [ ]:
!pip -q install "transformers==4.49.0" "tokenizers==0.21.0" \
                "accelerate>=0.34.0" "safetensors>=0.4.3" "huggingface-hub>=0.25.2" "sentence-transformers>=2.2" \
                "chromadb" "langchain>=0.2" "langchain-community" "langchain-text-splitters" \
                "bitsandbytes>=0.43.1,<0.44"  # ok to fail on CPU; we handle fallback



In [ ]:
!pip install -U torch torchvision torchaudio

In [ ]:
from huggingface_hub import login
login(token="")  # or use Kaggle Secrets


In [ ]:
#!/usr/bin/env python3
import json
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

DATA = Path("/kaggle/input/bhagwad-gita-json/Bhagwad_Gita.json")
DB_DIR = Path("gita_db")
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

def to_str(x: Any) -> str:
    if isinstance(x, str): return x
    if x is None: return ""
    if isinstance(x, (dict, list)): return json.dumps(x, ensure_ascii=False)
    return str(x)

def join_nonempty(parts: List[str], sep: str = "\n") -> str:
    return sep.join([p for p in parts if isinstance(p, str) and p.strip()])

def build_content(row: Dict[str, Any]) -> str:
    parts = [
        f"ID: {to_str(row.get('id'))}",
        f"Chapter {to_str(row.get('chapter'))}, Verse {to_str(row.get('verse'))}",
        f"Sanskrit: {to_str(row.get('sanskrit'))}",
        f"Translation: {to_str(row.get('translation'))}",
        f"Summary: {to_str(row.get('summary'))}",
        f"Context: {to_str(row.get('therapy_context'))}",
        f"Keywords: {to_str(row.get('keywords'))}",
    ]
    content = join_nonempty(parts, "\n")
    assert isinstance(content, str)
    return content

def load_dataframe():
    with DATA.open("r", encoding="utf-8") as f:
        raw = json.load(f)
    if isinstance(raw, list):
        return pd.DataFrame(raw)
    if isinstance(raw, dict):
        for key in ["data", "verses", "items", "records"]:
            if key in raw and isinstance(raw[key], list):
                return pd.DataFrame(raw[key])
    raise ValueError("Unsupported JSON structure")

def validate_docs(docs: List[Document]):
    for i, d in enumerate(docs):
        if not isinstance(d.page_content, str):
            raise TypeError(f"Doc[{i}] page_content not str")
        _ = d.page_content.replace("", "")
        if d.metadata is None: d.metadata = {}
        for k, v in list(d.metadata.items()):
            if not isinstance(v, (str, int, float, bool)) and v is not None:
                d.metadata[k] = to_str(v)

def build_documents(df: pd.DataFrame) -> List[Document]:
    df = df.fillna("")
    docs: List[Document] = []
    for _, r in df.iterrows():
        row = r.to_dict()
        docs.append(Document(
            page_content=build_content(row),
            metadata={
                "id": to_str(row.get("id")),
                "chapter": to_str(row.get("chapter")),
                "verse": to_str(row.get("verse")),
            }
        ))
    validate_docs(docs)
    return docs

def main():
    if not DATA.exists():
        raise FileNotFoundError(f"Input not found: {DATA.resolve()}")
    df = load_dataframe()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=700, chunk_overlap=100, separators=["\n\n", "\n", "।", ".", " "]
    )
    docs = build_documents(df)
    splits = splitter.split_documents(docs)
    embeddings = HuggingFaceEmbeddings(model_name=EMB_MODEL)
    db = Chroma.from_documents(
        splits, embedding=embeddings,
        persist_directory=str(DB_DIR), collection_name="gita"
    )
    db.persist()
    print(f"Indexed {len(splits)} chunks into {DB_DIR.resolve()}")

if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
import os, torch
from typing import List, Dict, Any
from operator import itemgetter

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.runnables import RunnableParallel, RunnableLambda

MODEL_NAME = os.environ.get("MISTRAL_LOCAL_MODEL", "mistralai/Mistral-7B-Instruct-v0.2")
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
DB_DIR = "gita_db"

def load_model(model_name: str):
    use_4bit = True
    bnb_cfg = None
    if use_4bit:
        try:
            bnb_cfg = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.bfloat16
            )
        except Exception:
            bnb_cfg = None

    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    if bnb_cfg is not None:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            quantization_config=bnb_cfg,
            device_map="auto"
        )
    else:
        # Fallback (needs more VRAM)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            device_map="auto"
        )
    return tok, model

tokenizer, model = load_model(MODEL_NAME)

embeddings = HuggingFaceEmbeddings(model_name=EMB_MODEL)
vectordb = Chroma(
    persist_directory=DB_DIR,
    embedding_function=embeddings,
    collection_name="gita"
)
retriever = vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 4, "lambda_mult": 0.5})

SYSTEM = (
    "You are a Bhagavad Gita philosophy assistant.\n"
    "Answer with clear, compassionate reasoning grounded ONLY in the provided context.\n"
    "If unclear from context, say you don't know and offer closest relevant verses.\n\n"
    "CITATIONS:\n"
    "- After claims that depend on the text, add inline citations like (Ch 2:V 47 | BG-2-47).\n"
    "- Cite multiple verses if needed. Never invent verses/content.\n\n"
    "STYLE: Be concise (3–6 short paragraphs)."
)

def _s(x: Any) -> str:
    return x if isinstance(x, str) else ("" if x is None else str(x))

def format_docs(docs):
    blocks = []
    for d in docs:
        m = d.metadata or {}
        cid = _s(m.get("id")); ch = _s(m.get("chapter")); vs = _s(m.get("verse"))
        blocks.append(f"[{cid}] Ch {ch} | V {vs}\n{_s(d.page_content)}")
    return "\n\n---\n\n".join(blocks)

# Only the string question enters the retriever (prevents dict.replace issues)
question_only = RunnableLambda(lambda x: x["question"])
inputs = RunnableParallel(
    context=question_only | retriever | format_docs,
    question=itemgetter("question"),
    chat_history=itemgetter("chat_history")
)

def mistral_generate(payload: Dict[str, Any]) -> str:
    # Build Mistral chat template using tokenizer (supports apply_chat_template)
    messages = [{"role":"system","content":SYSTEM}]

    for turn in payload.get("chat_history", []):
        role = turn.get("role","user")
        content = _s(turn.get("content",""))
        if content: messages.append({"role": role, "content": content})

    user_msg = (
        f"User question: {payload['question']}\n\n"
        f"Use these retrieved notes:\n{payload['context']}\n\n"
        f"Now answer faithfully with citations."
    )
    messages.append({"role":"user","content":user_msg})

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs_tok = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs_tok,
            max_new_tokens=512,
            do_sample=False,
            temperature=0.2,
            top_p=0.95,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    gen_ids = out[0][inputs_tok["input_ids"].shape[1]:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return text.strip()

chain = inputs | RunnableLambda(mistral_generate)

def main():
    print("Bhagavad Gita RAG (Transformers + Mistral-7B-Instruct). Type 'exit' to quit.")
    chat_history: List[Dict[str, str]] = []
    while True:
        try:
            q = input("\nYou: ").strip()
        except (EOFError, KeyboardInterrupt):
            break
        if not q or q.lower() in {"exit","quit"}:
            break
        res = chain.invoke({"question": q, "chat_history": chat_history})
        print("\nAssistant:", res)
        chat_history = (chat_history + [{"role":"user","content":q},
                                        {"role":"assistant","content":res}])[-12:]

if __name__ == "__main__":
    main()
